# **Protein Embeddings with ESM-2**

Welcome to the Neural Network practice Day 6!

In this session, we will explore **Evolutionary Scale Modeling (ESM)** embeddings. These are powerful representations of protein sequences learned by transformer models trained on billions of protein sequences. We will use Hugging Face `transformers` to load ESM models and analyze protein similarities using embedding distances.

## **Module Imports**

To work with protein embeddings, we need:
- `torch`: The core PyTorch library
- `transformers`: Hugging Face library to load ESM models
- `scikit-learn`: For dimensionality reduction (PCA)
- `matplotlib` & `seaborn`: For static and 2D/3D plotting
- `plotly`: For interactive figures

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch
from transformers import AutoTokenizer, EsmModel
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import umap
import plotly.graph_objects as go

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


---

## **Understanding ESM Embeddings**

ESM (Evolutionary Scale Modeling) models are "Protein Language Models". Just as BERT or GPT understand human language, ESM understands the "language of life"—amino acid sequences.

### **How it works:**
1. **Tokenization**: A protein sequence (e.g., `MKV...`) is broken into individual amino acids (tokens).
2. **Transformer Layers**: The model processes these tokens, capturing dependencies between amino acids.
3. **Embedding Extraction**: We take the internal mathematical representation (a high-dimensional vector aka tensor) from the model's layers. This vector represents the "meaning" of the protein in a biological/functional space based on what the model learned (it is not always true and can make mistakes as chatGPT can make).

### **The Embedding Vector:**
- A single protein is compressed into a vector of fixed size (e.g., 320, 480, or 1280 dimensions depending on the model size).

---

## **Loading ESM-2 from Hugging Face**

We will use the `esm2_t6_8M_UR50D` model. It is a lightweight version (8 million parameters) that runs quickly even on a CPU.

In [6]:
# Load the tokenizer and model
model_name = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(model_name).to(device)

def get_protein_embedding(sequence):
    """
    Converts a protein sequence into a single 320-dimensional embedding vector.
    We use 'Mean Pooling' (averaging all amino acid vectors).
    """
    # 1. Tokenize and move to device
    inputs = tokenizer(sequence, return_tensors="pt").to(device)
    
    # 2. Forward pass through the model
    with torch.no_grad():
        outputs = model(**inputs)
    
    # 3. Extract the last hidden state (shape: [batch, sequence_length, embedding_dim])
    last_hidden_state = outputs.last_hidden_state
    
    # 4. Mean pooling (average across the sequence length dimension)
    embedding = last_hidden_state.mean(dim=1).cpu().numpy()
    
    return embedding.flatten()

# Test it
test_seq = "MKVLWAALLVTFLAGCQAKVEQAVETEPEPELRQQPRW"
emb = get_protein_embedding(test_seq)
print(f"Sequence length: {len(test_seq)}")
print(f"Embedding shape: {emb.shape}")

---

## **Cosine Similarity vs. Euclidean Distance**

When comparing protein embeddings, we usually choose between two metrics. Understanding their difference is crucial for biological interpretation.

### **1. Cosine Similarity**
Measures the **angle** between two vectors. It asks: *"Do these vectors point in the same direction?"*
- **Range**: -1 to 1 (usually 0 to 1 for embeddings).
- **Property**: It is **magnitude-invariant**. If you double the length of a vector but keep its direction, the cosine similarity remains exactly the same.
- **Biological Intuition**: Focuses on the *functional pattern* or *signature* of the protein, regardless of how "strong" or "long" the embedding is.

### **2. Euclidean Distance**
Measures the **straight-line distance** between two points. It asks: *"How far apart are these points in space?"*
- **Range**: 0 to $\infty$.
- **Property**: Sensitive to both **direction and magnitude**.
- **Biological Intuition**: Useful if the model uses vector length to encode information (like structural stability or confidence).

### **Why choose Cosine Similarity for Proteins?**
1. **Curse of Dimensionality**: In high-dimensional spaces (like ESM's 320D), Euclidean distance can become less meaningful as almost all points become "far apart".
2. **Pooling Artifacts**: Since we often average amino acid vectors, a longer protein might naturally have a slightly different magnitude than a shorter one. Cosine similarity ignores these "length" effects and focuses on the sequence signature.
3. **Standard Practice**: Most protein search tools (like FoldSeek or ESM-based searches) use Cosine Similarity because it better captures functional relationships.

---

## **Interactive Comparison: Angle vs. Magnitude**

Use the sliders below to see how changing a vector's **angle** and **magnitude** affects the two metrics differently.

In [ ]:
from plotly.subplots import make_subplots


def vec_from_angles(az_deg, el_deg, mag):
    az = np.radians(az_deg)
    el = np.radians(el_deg)
    return mag * np.array([np.cos(el) * np.cos(az), np.cos(el) * np.sin(az), np.sin(el)])


def update_plot(az_A, el_A, mag_A, az_B, el_B, mag_B):
    v_a_3d = vec_from_angles(az_A, el_A, mag_A)
    v_b_3d = vec_from_angles(az_B, el_B, mag_B)
    v_a_2d = np.array([v_a_3d[0], v_a_3d[1]])
    v_b_2d = np.array([v_b_3d[0], v_b_3d[1]])

    n_a = np.linalg.norm(v_a_3d)
    n_b = np.linalg.norm(v_b_3d)
    cos_sim = np.dot(v_a_3d, v_b_3d) / (n_a * n_b) if (n_a > 1e-10 and n_b > 1e-10) else 0.0
    euc_dist = np.linalg.norm(v_a_3d - v_b_3d)

    limit_2d = max(1.5, mag_A + 0.5, mag_B + 0.5)

    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "xy"}, {"type": "scene"}]],
        subplot_titles=("2D view (xy projection)", "3D view"),
        horizontal_spacing=0.1,
    )

    # --- 2D (xy projection) ---
    fig.add_trace(
        go.Scatter(
            x=[0, v_a_2d[0]],
            y=[0, v_a_2d[1]],
            mode="lines+markers",
            line=dict(color="blue", width=4),
            marker=dict(size=[6, 8]),
            name="A",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=[0, v_b_2d[0]],
            y=[0, v_b_2d[1]],
            mode="lines+markers",
            line=dict(color="red", width=4),
            marker=dict(size=[6, 8]),
            name="B",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=[v_a_2d[0], v_b_2d[0]],
            y=[v_a_2d[1], v_b_2d[1]],
            mode="lines",
            line=dict(color="green", width=2, dash="dash"),
            name="Euc dist",
        ),
        row=1,
        col=1,
    )

    n_a_2d = np.linalg.norm(v_a_2d)
    n_b_2d = np.linalg.norm(v_b_2d)
    if n_a_2d > 1e-10 and n_b_2d > 1e-10:
        angle_a = np.arctan2(v_a_2d[1], v_a_2d[0])

        # Signed shortest rotation from A -> B in [-pi, pi]
        signed_angle = np.arctan2(
            v_a_2d[0] * v_b_2d[1] - v_a_2d[1] * v_b_2d[0],
            np.dot(v_a_2d, v_b_2d),
        )
        angle_between = np.abs(signed_angle)

        min_norm = min(n_a_2d, n_b_2d)
        arc_r = np.clip(0.4 * min_norm, 0.15, 0.55)
        t = np.linspace(0.0, 1.0, 80)
        arc_angles = angle_a + t * signed_angle
        arc_x = arc_r * np.cos(arc_angles)
        arc_y = arc_r * np.sin(arc_angles)

        fig.add_trace(
            go.Scatter(
                x=arc_x,
                y=arc_y,
                mode="lines",
                line=dict(color="purple", width=3),
                name="angle θ",
            ),
            row=1,
            col=1,
        )

        mid = angle_a + 0.5 * signed_angle
        label_r = arc_r + 0.12
        fig.add_annotation(
            x=label_r * np.cos(mid),
            y=label_r * np.sin(mid),
            text=f"θ = {np.degrees(angle_between):.0f}°",
            showarrow=False,
            font=dict(color="purple", size=12),
            row=1,
            col=1,
        )

    # --- 3D ---
    fig.add_trace(
        go.Scatter3d(
            x=[0, v_a_3d[0]],
            y=[0, v_a_3d[1]],
            z=[0, v_a_3d[2]],
            mode="lines+markers",
            line=dict(color="blue", width=6),
            marker=dict(size=4),
            name="A (3D)",
            showlegend=False,
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter3d(
            x=[0, v_b_3d[0]],
            y=[0, v_b_3d[1]],
            z=[0, v_b_3d[2]],
            mode="lines+markers",
            line=dict(color="red", width=6),
            marker=dict(size=4),
            name="B (3D)",
            showlegend=False,
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter3d(
            x=[v_a_3d[0], v_b_3d[0]],
            y=[v_a_3d[1], v_b_3d[1]],
            z=[v_a_3d[2], v_b_3d[2]],
            mode="lines",
            line=dict(color="green", width=4, dash="dash"),
            name="Euc dist (3D)",
            showlegend=False,
        ),
        row=1,
        col=2,
    )

    axis_3d = [-2.0, 2.0]
    fig.update_xaxes(range=[-limit_2d, limit_2d], zeroline=True, showgrid=True, row=1, col=1)
    fig.update_yaxes(range=[-limit_2d, limit_2d], zeroline=True, showgrid=True, scaleanchor="x", scaleratio=1, row=1, col=1)

    fig.update_layout(
        title="Adjust Vector A and B below - 2D and 3D update live",
        autosize=True,
        height=430,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
        margin=dict(l=20, r=20, t=55, b=20),
        scene=dict(
            xaxis=dict(title="X", range=axis_3d),
            yaxis=dict(title="Y", range=axis_3d),
            zaxis=dict(title="Z", range=axis_3d),
            aspectmode="cube",
        ),
    )

    fig.add_annotation(
        x=0.02,
        y=0.98,
        xref="paper",
        yref="paper",
        text=f"cos θ = {cos_sim:.3f}",
        showarrow=False,
        bgcolor="wheat",
        bordercolor="gray",
        borderwidth=1,
        font=dict(size=13),
    )

    fig.show(config={"responsive": True, "displaylogo": False})

    fig_metrics = go.Figure(
        data=[
            go.Table(
                header=dict(values=["<b>Metric</b>", "<b>Value</b>"], fill_color="lightgray", align="left"),
                cells=dict(
                    values=[["Cosine similarity", "Euclidean distance"], [f"{cos_sim:.4f}", f"{euc_dist:.4f}"]],
                    align="left",
                ),
            )
        ]
    )
    fig_metrics.update_layout(title="Current metrics", autosize=True, height=120, margin=dict(t=35, b=10, l=20, r=20))
    fig_metrics.show(config={"responsive": True, "displaylogo": False})


# Defaults: A = (1,0,0) mag 1, B = 30° azimuth mag 1
az_A = widgets.IntSlider(min=0, max=360, step=5, value=0, description="Azimuth (°)")
el_A = widgets.IntSlider(min=-90, max=90, step=5, value=0, description="Elevation (°)")
mag_A = widgets.FloatSlider(min=0.1, max=2.5, step=0.1, value=1.0, description="Magnitude")
az_B = widgets.IntSlider(min=0, max=360, step=5, value=30, description="Azimuth (°)")
el_B = widgets.IntSlider(min=-90, max=90, step=5, value=0, description="Elevation (°)")
mag_B = widgets.FloatSlider(min=0.1, max=2.5, step=0.1, value=1.0, description="Magnitude")

col_A = widgets.VBox([widgets.HTML("<b>Vector A</b>"), az_A, el_A, mag_A], layout=widgets.Layout(width="220px"))
col_B = widgets.VBox([widgets.HTML("<b>Vector B</b>"), az_B, el_B, mag_B], layout=widgets.Layout(width="220px"))


def on_reset(b):
    az_A.value = 0
    el_A.value = 0
    mag_A.value = 1.0
    az_B.value = 30
    el_B.value = 0
    mag_B.value = 1.0


reset_btn = widgets.Button(description="Reset plot", button_style="info")
reset_btn.on_click(on_reset)

out = widgets.interactive_output(
    update_plot,
    {"az_A": az_A, "el_A": el_A, "mag_A": mag_A, "az_B": az_B, "el_B": el_B, "mag_B": mag_B},
)
display(
    widgets.VBox([
        widgets.HBox([col_A, col_B, widgets.VBox([reset_btn])]),
        out,
    ])
)


---

## **Embeddings and Dimensionality Reduction: PCA and UMAP**

After comparing distances, we now project high-dimensional embeddings into 2D so we can **see structure**.

- **PCA (Principal Component Analysis)** is a **linear** method.
  - It finds orthogonal directions (principal components) that explain the largest variance in the data.
  - Good for a fast global overview and easy interpretation.
- **UMAP** is a **non-linear** method.
  - It preserves local neighborhoods, so points that are close in the original embedding space tend to stay close in 2D.
  - Good for visualizing local clusters and manifold-like structure.

### **Small toy example (synthetic embeddings)**
In the next cell, we build fake high-dimensional embeddings with three groups, then project them with both PCA and UMAP to compare how the structure appears in 2D.

In [ ]:
# Small toy example: synthetic high-dimensional embeddings
# (keeps runtime short and concept clear)
rng = np.random.default_rng(42)

n_per_group = 20
embedding_dim = 64

# Three groups centered at different regions in 64D
group_a = rng.normal(loc=0.0, scale=0.6, size=(n_per_group, embedding_dim))
group_b = rng.normal(loc=2.5, scale=0.6, size=(n_per_group, embedding_dim))
group_c = rng.normal(loc=-2.0, scale=0.6, size=(n_per_group, embedding_dim))

X = np.vstack([group_a, group_b, group_c])
labels = np.array(["A"] * n_per_group + ["B"] * n_per_group + ["C"] * n_per_group)

# PCA projection
pca_2d_toy = PCA(n_components=2, random_state=42).fit_transform(X)

# UMAP projection
umap_2d_toy = umap.UMAP(n_components=2, n_neighbors=10, min_dist=0.15, random_state=42).fit_transform(X)

# Plot side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
color_map = {"A": "tab:blue", "B": "tab:orange", "C": "tab:green"}

for g in ["A", "B", "C"]:
    idx = labels == g
    axes[0].scatter(pca_2d_toy[idx, 0], pca_2d_toy[idx, 1], label=f"Group {g}", alpha=0.8, color=color_map[g])
axes[0].set_title("PCA (toy embeddings)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].grid(alpha=0.3)
axes[0].legend()

for g in ["A", "B", "C"]:
    idx = labels == g
    axes[1].scatter(umap_2d_toy[idx, 0], umap_2d_toy[idx, 1], label=f"Group {g}", alpha=0.8, color=color_map[g])
axes[1].set_title("UMAP (toy embeddings)")
axes[1].set_xlabel("UMAP-1")
axes[1].set_ylabel("UMAP-2")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

---

# **Question 1: Measuring Protein Proximity**

**YOUR TASK**: 

Below are three protein sequences. Your goal is to determine their relationship using ESM embeddings.

1. Calculate their ESM embeddings.
2. Calculate the **Cosine Similarity** between `seq_a` and `seq_b` and `seq_c`.


In [ ]:
# ============================================================
# EXERCISE : Calculate Distances
# ============================================================

#PDB_ID: 104M
seq_a = "VLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRFKHLKTEAEMKASEDLKKHGVTVLTALGAILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISEAIIHVLHSRHPGDFGADAQGAMNKALELFRKDIAAKYKELGYQG"
#PDB_ID: 1MWC
seq_b = "GLSDGEWQLVLNVWGKVEADVAGHGQEVLIRLFKGHPETLEKFDKFKHLKSEDEMKASEDLKKHGNTVLTALGGILKKKGHHEAELTPLAQSHATKHKIPVKYLEFISEAIIQVLQSKHPGDFGADAQGAMSKALELFRNDMAAKYKELGFQG"
#PDB ID: 10LE
seq_c = "MHHHHHHHHAHMKRYLGGLDVFRYIGPGLLVTVGFIDPGNWASNFAAGSEFGYSLLWVVTLSTIMLIILQHNVAHLGIVTGLCLSEAATQYTPKWVSRPILGTAVLASISTSLAEILGGAIALEMLLDIPIVWGAVLTTVFVSIMLFTNSYKKIERSIIAFVSVIGLSFIYELFLVDIDWPMAVEGWVTPAIPKGSMLIIMSVLGAVVMPHNLFLHSEVIQSHEYNKQDTASIKKVLKYELFDTLFSMIIGWAINSAMILLAAATFFKSGIQVEELQQAKSLLEPLLGSNAAIVFALALLMAGISSTITSGMAAGSIFAGIFGESYHIKDSHSQVGVILSLGIALLLIFFIGDPFKGLIISQMVLSIQLPFTVFLQVGLTSSRKVMGDYVNSKWSTFVLYTIAVIVTVLNIMLLFS"

# TODO: Extract embeddings for all three proteins


# TODO: Calculate Cosine Similarity between A and B

# TODO: Calculate Cosine Similarity between A and C

# TODO: Calculate Cosine Similarity between B and C


# TODO: Print the results and compare them

---

# **Question 2: Visualizing Protein Populations**

PCA and UMAP both reduce high-dimensional embeddings to 2D so we can visualize patterns.

- **PCA (Principal Component Analysis)** is a **linear** method. It finds orthogonal directions (principal components) that explain the largest variance in the data.
- **UMAP** is a **non-linear** method. It preserves local neighborhoods, so proteins that are close in the original embedding space tend to stay close in 2D.

**Small practical example in this notebook:**
1. Take a small subset of proteins.
2. Extract ESM embeddings.
3. Project with PCA and UMAP.
4. Compare the two 2D plots.

### **Protein Populations:**

In [ ]:
protein_data = [
    {"pdb_id": "1MBN", "seq": "VLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRFKHLKTEAEMKASEDLKKHGVTVLTALGAILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISEAIIHVLHSRHPGDFGADAQGAMNKALELFRKDIAAKYKELGYQG"},
    {"pdb_id": "1PMB", "seq": "GLSDGEWQLVLNVWGKVEADVAGHGQEVLIRLFKGHPETLEKFDKFKHLKSEDEMKASEDLKKHGNTVLTALGGILKKKGHHEAELTPLAQSHATKHKIPVKYLEFISEAIIQVLQSKHPGDFGADAQGAMSKALELFRNDMAAKYKELGFQG"},
    {"pdb_id": "1AZI", "seq": "GLSDGEWQQVLNVWGKVEADIAGHGQEVLIRLFTGHPETLEKFDKFKHLKTEAEMKASEDLKKHGTVVLTALGGILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISDAIIHVLHSKHPGDFGADAQGAMTKALELFRNDIAAKYKELGFQG"},
    {"pdb_id": "2DHB", "seq": "VQLSGEEKAAVLALWDKVNEEEVGGEALGRLLVVYPWTQRFFDSFGDLSNPGAVMGNPKVKAHGKKVLHSFGEGVHHLDNLKGTFAALSELHCDKLHVDPENFRLLGNVLALVVARHFGKDFTPELQASYQKVVAGVANALAHKYH"},
    {"pdb_id": "2HHB", "seq": "VLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"},
    {"pdb_id": "1ECO", "seq": "LSADQISTVQASFDKVKGDPVGILYAVFKADPSIMAKFTQFAGKDLESIKGTAPFETHANRIVGFFSKIIGELPNIEADVNTFVASHKPRGVTHDQLNNFRAGFVSYMKAHTDFAGAEAAWGATLDTFFGMIFSKM"},
    {"pdb_id": "1VHB", "seq": "MLDQQTINIIKATVPVLKEHGVTITTTFYKNLFAKHPEVRPLFDMGRQESLEQPKALAMTVLAAAQNIENLPAILPAVKKIAVKHCQAGVAAAHYPIVGQELLGAIKEVLGDAATDDILDAWGKAYGVIADVFIQVEADLYAQAVE"},
    {"pdb_id": "1ITH", "seq": "GLTAAQIKAIQDHWFLNIKGCLQAAADSIFFKYLTAYPGDLAFFHKFSSVPLYGLRSNPAYKAQTLTVINYLDKVVDALGGNAGALMKAKVPSHDAMGITPKHFGQLLKLVGGVFQEEFSADPTTVAAWGDAAGVLVAAMK"},
    {"pdb_id": "1FLP", "seq": "SLEAAQKSNVTSSWAKASAAWGTAGPEFFMALFDAHDDVFAKFSGLFSGAAKGTVKNTPEMAAQAQSFKGLVSNWVDNLDNAGALEGQCKTFAANHKARGISAGQLEAAFKVLSGFMKSYGGDEGAWTAVAGALMGEIEPDM"},
    {"pdb_id": "1MYG", "seq": "GLSDGEWQLVLNVWGKVEADVAGHGQEVLIRLFKGHPETLEKFDKFKHLKSEDEMKASEDLKKHGNTVLTALGGILKKKGHHEAELTPLAQSHATKHKIPVKYLEFISEAIIQVLQSKHPGDFGADAQGAMSKALELFRNDMAAKYKELGFQG"},
    {"pdb_id": "1A6M", "seq": "VLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRFKHLKTEAEMKASEDLKKHGVTVLTALGAILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISEAIIHVLHSRHPGDFGADAQGAMNKALELFRKDIAAKYKELGY"},
    {"pdb_id": "1OUT", "seq": "VEWTDAEKSTISAVWGKVNIDEIGPLALARVLIVYPWTQRYFGSFGNVSTPAAIMGNPKVAAHGKVVCGALDKAVKNMGNILATYKSLSETHANKLFVDPDNFRVLADVLTIVIAAKFGASFTPEIQATWQKFMKVVVAAMGSRYF"},
    {"pdb_id": "1S60", "seq": "MGSSHHHHHHSSGLVPRGSHMDIRQMNKTHLEHWRGLRKQLWPGHPDDAHLADGEEILQADHLASFIAMADGVAIGFADASIRHDYVNGCDSSPVVFLEGIFVLPSFRQRGVAKQLIAAVQRWGTNKGCREMASDTSPENTISQKVHQALGFEETERVIFYRKRC"},
    {"pdb_id": "2SRC", "seq": "MVTTFVALYDYESRTETDLSFKKGERLQIVNNTEGDWWLAHSLSTGQTGYIPSNYVAPSDSIQAEEWYFGKITRRESERLLLNAENPRGTFLVRESETTKGAYCLSVSDFDNAKGLNVKHYKIRKLDSGGFYITSRTQFNSLQQLVAYYSKHADGLCHRLTTVCPTSKPQTQGLAKDAWEIPRESLRLEVKLGQGCFGEVWMGTWNGTTRVAIKTLKPGTMSPEAFLQEAQVMKKLRHEKLVQLYAVVSEEPIYIVTEYMSKGSLLDFLKGETGKYLRLPQLVDMAAQIASGMAYVERMNYVHRDLRAANILVGENLVCKVADFGLARLIEDNEYTARQGAKFPIKWTAPEAALYGRFTIKSDVWSFGILLTELTTKGRVPYPGMVNREVLDQVERGYRMPCPPECPESLHDLMCQCWRKEPEERPTFEYLQAFLEDYFTSTEPQYQPGENL"},
    {"pdb_id": "1FMK", "seq": "MVTTFVALYDYESRTETDLSFKKGERLQIVNNTEGDWWLAHSLSTGQTGYIPSNYVAPSDSIQAEEWYFGKITRRESERLLLNAENPRGTFLVRESETTKGAYCLSVSDFDNAKGLNVKHYKIRKLDSGGFYITSRTQFNSLQQLVAYYSKHADGLCHRLTTVCPTSKPQTQGLAKDAWEIPRESLRLEVKLGQGCFGEVWMGTWNGTTRVAIKTLKPGTMSPEAFLQEAQVMKKLRHEKLVQLYAVVSEEPIYIVTEYMSKGSLLDFLKGETGKYLRLPQLVDMAAQIASGMAYVERMNYVHRDLRAANILVGENLVCKVADFGLARLIEDNEYTARQGAKFPIKWTAPEAALYGRFTIKSDVWSFGILLTELTTKGRVPYPGMVNREVLDQVERGYRMPCPPECPESLHDLMCQCWRKEPEERPTFEYLQAFLEDYFTSTEPQYQPGENL"},
    {"pdb_id": "3CS9", "seq": "GAMDPSPNYDKWEMERTDITMKHKLGGGQYGEVYEGVWKKYSLTVAVKTLKEDTMEVEEFLKEAAVMKEIKHPNLVQLLGVCTREPPFYIITEFMTYGNLLDYLRECNRQEVNAVVLLYMATQISSAMEYLEKKNFIHRDLAARNCLVGENHLVKVADFGLSRLMTGDTYTAHAGAKFPIKWTAPESLAYNKFSIKSDVWAFGVLLWEIATYGMSPYPGIDLSQVYELLEKDYRMERPEGCPEKVYELMRACWQWNPSDRPSFAEIHQAFETMFQES"},
    {"pdb_id": "1OPK", "seq": "GAMDPSEALQRPVASDFEPQGLSEAARWNSKENLLAGPSENDPNLFVALYDFVASGDNTLSITKGEKLRVLGYNHNGEWCEAQTKNGQGWVPSNYITPVNSLEKHSWYHGPVSRNAAEYLLSSGINGSFLVRESESSPGQRSISLRYEGRVYHYRINTASDGKLYVSSESRFNTLAELVHHHSTVADGLITTLHYPAPKRNKPTIYGVSPNYDKWEMERTDITMKHKLGGGQYGEVYEGVWKKYSLTVAVKTLKEDTMEVEEFLKEAAVMKEIKHPNLVQLLGVCTREPPFYIITEFMTYGNLLDYLRECNRQEVSAVVLLYMATQISSAMEYLEKKNFIHRNLAARNCLVGENHLVKVADFGLSRLMTGDTYTAHAGAKFPIKWTAPESLAYNKFSIKSDVWAFGVLLWEIATYGMSPYPGIDLSQVYELLEKDYRMERPEGCPEKVYELMRACWQWNPSDRPSFAEIHQAFETMFQESSISDEVEKELGKRGT"},
    {"pdb_id": "1IRK", "seq": "VFPSSVFVPDEWEVSREKITLLRELGQGSFGMVYEGNARDIIKGEAETRVAVKTVNESASLRERIEFLNEASVMKGFTCHHVVRLLGVVSKGQPTLVVMELMAHGDLKSYLRSLRPEAENNPGRPPPTLQEMIQMAAEIADGMAYLNAKKFVHRDLAARNCMVAHDFTVKIGDFGMTRDIYETDYYRKGGKGLLPVRWMAPESLKDGVFTTSSDMWSFGVVLWEITSLAEQPYQGLSNEQVLKFVMDGGYLDQPDNCPERVTDLMRMCWQFNPKMRPTFLEIVNLLKDDLHPSFPEVSFFHSEENK"},
    {"pdb_id": "1FGK", "seq": "MVAGVSEYELPEDPRWELPRDRLVLGKPLGEGAFGQVVLAEAIGLDKDKPNRVTKVAVKMLKSDATEKDLSDLISEMEMMKMIGKHKNIINLLGACTQDGPLYVIVEYASKGNLREYLQARRPPGLEYSYNPSHNPEEQLSSKDLVSCAYQVARGMEYLASKKCIHRDLAARNVLVTEDNVMKIADFGLARDIHHIDYYKKTTNGRLPVKWMAPEALFDRIYTHQSDVWSFGVLLWEIFTLGGSPYPGVPVEELFKLLKEGHRMDKPSNCTNELYMMMRDCWHAVPSQRPTFKQLVEDLDRIVALTSNQE"},
    {"pdb_id": "2PHK", "seq": "GFYENYEPKEILGRGVSSVVRRCIHKPTCKEYAVKIIDVTGGGSFSAEEVQELREATLKEVDILRKVSGHPNIIQLKDTYETNTFFFLVFDLMKKGELFDYLTEKVTLSEKETRKIMRALLEVICALHKLNIVHRDLKPENILLDDDMNIKLTDFGFSCQLDPGEKLREVCGTPSYLAPEIIECSMNDNHPGYGKEVDMWSTGVIMYTLLAGSPPFWHRKQMLMLRMIMSGNYQFGSPEWDDYSDTVKDLVSRFLVVQPQKRYTAEEALAHPFFQQY"},
    {"pdb_id": "1CKP", "seq": "MENFQKVEKIGEGTYGVVYKARNKLTGEVVALKKIRLDTETEGVPSTAIREISLLKELNHPNIVKLLDVIHTENKLYLVFEFLHQDLKKFMDASALTGIPLPLIKSYLFQLLQGLAFCHSHRVLHRDLKPQNLLINTEGAIKLADFGLARAFGVPVRTYTHEVVTLWYRAPEILLGCKYYSTAVDIWSLGCIFAEMVTRRALFPGDSEIDQLFRIFRTLGTPDEVVWPGVTSMPDYKPSFPKWARQDFSKVVPPLDEDGRSLLSQMLHYDPNKRISAKAALAHPFFQDVTKPVPHLRL"},
    {"pdb_id": "1ATP", "seq": "GNAAAAKKGSEQESVKEFLAKAKEDFLKKWETPSQNTAQLDQFDRIKTLGTGSFGRVMLVKHKESGNHYAMKILDKQKVVKLKQIEHTLNEKRILQAVNFPFLVKLEFSFKDNSNLYMVMEYVAGGEMFSHLRRIGRFSEPHARFYAAQIVLTFEYLHSLDLIYRDLKPENLLIDQQGYIQVTDFGFAKRVKGRTWTLCGTPEYLAPEIILSKGYNKAVDWWALGVLIYEMAAGYPPFFADQPIQIYEKIVSGKVRFPSHFSSDLKDLLRNLLQVDLTKRFGNLKNGVNDIKNHKWFATTDWIAIYQRKVEAPFIPKFKGPGDTSNFDDYEEEEIRVSINEKCGKEFTEF"},
    {"pdb_id": "3P5O", "seq": "SMNPPPPETSNPNKPKRQTNQLQYLLRVVLKTLWKHQFAWPFQQPVDAVKLNLPDYYKIIKTPMDMGTIKKRLENNYYWNAQECIQDFNTMFTNCYIYNKPGDDIVLMAEALEKLFLQKINELPTEE"},
    {"pdb_id": "2F4J", "seq": "GMSPNYDKWEMERTDITMKHKLGGGQYGEVYEGVWKKYSLTVAVKTLKEDTMEVEEFLKEAAVMKEIKHPNLVQLLGVCTREPPFYIITEFMTYGNLLDYLRECNRQEVNAVVLLYMATQISSAMEYLEKKNFIHRDLAARNCLVGENHLVKVADFGLSRLMTGDTYTAPAGAKFPIKWTAPESLAYNKFSIKSDVWAFGVLLWEIATYGMSPYPGIDLSQVYELLEKDYRMERPEGCPEKVYELMRACWQWNPSDRPSFAEIHQAFETMFQESSISDEVEKELGKQ"},
    {"pdb_id": "1M17", "seq": "GSHMASGEAPNQALLRILKETEFKKIKVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDEEDMDDVVDADEYLIPQQG"},
    {"pdb_id": "2GS7", "seq": "GAMGEAPNQALLRILKETEFKKIKVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMRKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDEEDMDDVVDADEYLIPQQG"},
    {"pdb_id": "2CMW", "seq": "SMRVGKKIGCGNFGELRLGKNLYTNEYVAIKLEPIKSRAPQLHLEYRFYKQLGSAGEGLPQVYYFGPCGKYNAMVLELLGPSLEDLFDLCDRTFTLKTVLMIAIQLLSRMEYVHSKNLIYRDVKPENFLIGRQGNKKEHVIHIIDFGLAKEYIDPETKKHIPYREHKSLTGTARYMSINTHLGKEQSRRDDLEALGHMFMYFLRGSLPWQGLKADTLKERYQKIGDTKRNTPIEALCENFPEEMATYLRYVRRLDFFEKPDYEYLRTLFTDLFEKKGYTFDYAYDWVGRPIPTPVGSVHVDSGASAITRE"},
    {"pdb_id": "1HRC", "seq": "GDVEKGKKIFVQKCAQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGFTYTDANKNKGITWKEETLMEYLENPKKYIPGTKMIFAGIKKKTEREDLIAYLKKATNE"},
    {"pdb_id": "1YCC", "seq": "TEFKAGSAKKGATLFKTRCLQCHTVEKGGPHKVGPNLHGIFGRHSGQAEGYSYTDANIKKNVLWDENNMSEYLTNPKKYIPGTKMAFGGLKKEKDRNDLITYLKKACE"},
    {"pdb_id": "3ZCF", "seq": "GDVEKGKKIFIMKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGYSYTAANKNKGIIWGEDTLMEYLENPKKYIPGTKMIFVGIKKKEERADLIAYLKKATNE"},
    {"pdb_id": "1J3S", "seq": "GDVEKGKKIFIMKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGYSYTAANKNKGIIWGEDTLMEYLENPKKYIPGTKMIFVGIKKKEERADLIAYLKKATNE"},
    {"pdb_id": "1CYC", "seq": "GDVAKGKKTFVQKCAQCHTVENGGKHKVGPNLWGLFGRKTGQAEGYSYTDANKSKGIVWNENTLMEYLENPKKYIPGTKMIFAGIKKKGERQDLVAYLKSATS"},
    {"pdb_id": "5CYT", "seq": "GDVAKGKKTFVQKCAQCHTVENGGKHKVGPNLWGLFGRKTGQAEGYSYTDANKSKGIVWNNDTLMEYLENPKKYIPGTKMIFAGIKKKGERQDLVAYLKSATS"},
    {"pdb_id": "1C75", "seq": "VDAEAVVQQKCISCHGGDLTGASAPAIDKAGANYSEEEILDIILNGQGGMPGGIAKGAEAEAVAAWLAEKK"},
    {"pdb_id": "1AYG", "seq": "NEQLAKQKGCMACHDLKAKKVGPAYADVAKKYAGRKDAVDYLAGKIKKGGSGVWGSVPMPPQNVTDAEAKQLAQWILSIK"},
    {"pdb_id": "1C2R", "seq": "GDAAKGEKEFNKCKTCHSIIAPDGTEIVKGAKTGPNLYGVVGRTAGTYPEFKYKDSIVALGASGFAWTEEDIATYVKDPGAFLKEKLDDKKAKTGMAFKLAKGGEDVAAYLASVVK"},
    {"pdb_id": "1C52", "seq": "QADGAKIYAQCAGCHQQNGQGIPGAFPPLAGHVAEILAKEGGREYLILVLLYGLQGQIEVKGMKYNGVMSSFAQLKDEEIAAVLNHIATAWGDAKKVKGFKPFTAEEVKKLRAKKLTPQQVLAERKKLGLK"},
    {"pdb_id": "1CC5", "seq": "GGGARSGDDVVAKYCNACHGTGLLNAPKVGDSAAWKTRADAKGGLDGLLAQSLSGLNAMPPKGTCADCSDDELKAAIGKMSGL"},
    {"pdb_id": "1M60", "seq": "GDVEKGKKIFVQKCAQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGFTYTDANKNKGITWKEETLMEYLENPKKYIPGTKMIFAGIKKKTEREDLIAYLKKATNE"},
    {"pdb_id": "2B4Z", "seq": "GDVEKGKKIFVQKCAQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGFSYTDANKNKGITWGEETLMEYLENPKKYIPGTKMIFAGIKKKGEREDLIAYLKKATNE"},
    {"pdb_id": "1YEA", "seq": "AKESTGFKPGSAKKGATLFKTRCQQCHTIEEGGPNKVGPNLHGIFGRHSGQVKGYSYTDANINKNVKWDEDSMSEYLTNPKKYIPGTKMAFAGLKKEKDRNDLITYMTKAAK"},
    {"pdb_id": "1GFL", "seq": "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"},
]



In [ ]:
# TODO: Extract embeddings for all proteins in the list


# TODO: Run PCA


# TODO: Create a PCA Scatter plot


In [ ]:
# TODO: Run UMAP


# TODO: Create a UMAP Scatter plot